In [ ]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
load_dotenv(".env")

In [2]:
from benchmark_datasets import OpenMLBenchmark

In [3]:
bench = OpenMLBenchmark()
print("Benchmark suite (20 datasets):")
print(bench.list().to_string(index=False))

ds = bench.load("wdbc")
print(f"\nLoaded {ds.name!r}: X={ds.X.shape} ({ds.X.dtype}), y={ds.y.shape}")
print(f"classes: {ds.n_classes}, feature count: {len(ds.feature_names)}")

Benchmark suite (20 datasets):
                  name  data_id           task  n_rows                                                                          note
           tic-tac-toe       50 classification     958                                       XOR-like win patterns, pure interaction
      monks-problems-2      334 classification     601                                                 synthetic XOR / parity target
                 sonar       40 classification     208                                60 correlated sonar bands, non-linear boundary
            ionosphere       59 classification     351                                                      radar signal, non-linear
               vehicle       54 classification     846                                                   4-class silhouette geometry
                  wdbc     1510 classification     569                                breast cancer, non-linear feature interactions
              diabetes       37 classi

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from aug_dataset import aug_dataset

In [12]:
from sklearn.metrics import accuracy_score
sss = StratifiedShuffleSplit(1,train_size=8)
train_index, test_index = next(sss.split(ds.X, ds.y))
X_train = ds.X[train_index]
y_train = ds.y[train_index]
X_test = ds.X[test_index]
y_test = ds.y[test_index]

print('Num labeled samples:', len(X_train))

clf = TabPFNClassifier()
clf.fit(X_train, y_train)
base_pfn_pred = clf.predict(X_test)
print('Base tabPFN acc: ', accuracy_score(y_test,base_pfn_pred))

X_aug = aug_dataset(16, X_train, 0.8, 1, 0.5) 

X_aug_conf_indx = abs(0.5-clf.predict_proba(X_aug)[:, 0])>0.2
X_aug_conf = X_aug[X_aug_conf_indx]
print('Num augmented samples:', len(X_aug))
print('Num confident augmented samples:', len(X_aug_conf))

y_aug = clf.predict(X_aug)
y_aug_conf = clf.predict(X_aug_conf)

y_sl_aug = clf.predict_proba(X_aug)[:,1]

print('done')

Num labeled samples: 8
Base tabPFN acc:  0.9393939393939394
Num augmented samples: 136
Num confident augmented samples: 135
done


In [13]:
results = [{'name': 'TabPFN', 'base': accuracy_score(y_test,base_pfn_pred)}]

for n_estim in [1,2,3,5,10,100]:
    rfr = RandomForestClassifier(n_estimators=n_estim)
    rfr.fit(X_train, y_train)
    base_rfr_pred = rfr.predict(X_test)
    print(F'RFR (n_estim={n_estim}) base acc: ', accuracy_score(y_test,base_rfr_pred))
    

    rfr_aug = RandomForestClassifier(n_estimators=n_estim)
    rfr_aug.fit(X_aug, y_aug)
    rfr_aug_pred = rfr_aug.predict(X_test)
    print(F'RFR (n_estim={n_estim}) aug acc: ', accuracy_score(y_test,rfr_aug_pred))

    rfr_aug_conf = RandomForestClassifier(n_estimators=n_estim)
    rfr_aug_conf.fit(X_aug_conf, y_aug_conf)
    rfr_aug_conf_pred = rfr_aug_conf.predict(X_test)
    print(F'RFR (n_estim={n_estim}) aug conf acc: ', accuracy_score(y_test,rfr_aug_conf_pred))

    rfr_sl_aug = RandomForestRegressor(n_estimators=n_estim)
    rfr_sl_aug.fit(X_aug, y_sl_aug)
    rfr_sl_aug_pred = rfr_sl_aug.predict(X_test)>0.5
    print(F'RFR (n_estim={n_estim}) aug sl acc: ', accuracy_score(y_test,rfr_sl_aug_pred))

    results.append({'name': f'RFR n_estim={n_estim}', 
                    'base': accuracy_score(y_test,base_rfr_pred),
                    'aug': accuracy_score(y_test, rfr_aug_pred),
                    'aug confident': accuracy_score(y_test, rfr_aug_conf_pred),
                    'solf-labels aug': accuracy_score(y_test,rfr_sl_aug_pred)})

for hid_size in [(), (8,),(32,),(128,), (16,16,), (32,32), (16,16,16,), (67,67,67,)]:
    mlp = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp.fit(X_train, y_train)
    base_mlp_pred = mlp.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) base acc: ', accuracy_score(y_test,base_mlp_pred))

    mlp_aug = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_aug.fit(X_aug, y_aug)
    mlp_aug_pred= mlp_aug.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) aug acc: ', accuracy_score(y_test,mlp_aug_pred))

    mlp_aug_conf = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_aug_conf.fit(X_aug_conf, y_aug_conf)
    mlp_aug_conf_pred= mlp_aug_conf.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) aug conf acc: ', accuracy_score(y_test,mlp_aug_conf_pred))

    mlp_sl_aug = MLPRegressor(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_sl_aug.fit(X_aug, y_sl_aug)
    mlp_sl_aug_pred = mlp_sl_aug.predict(X_test) > 0.5
    print(f'MLP (hidden_l={hid_size}) soft-labels aug acc: ', accuracy_score(y_test,mlp_sl_aug_pred))

    results.append({'name': f'MLP hidden_l={hid_size}', 
                'base': accuracy_score(y_test,base_mlp_pred),
                'aug': accuracy_score(y_test, mlp_aug_pred),
                'aug confident': accuracy_score(y_test, mlp_aug_conf_pred),
                'solf-labels aug': accuracy_score(y_test, mlp_sl_aug_pred)})

results_df = pd.DataFrame(results)
results_df

RFR (n_estim=1) base acc:  0.8110516934046346
RFR (n_estim=1) aug acc:  0.8841354723707665
RFR (n_estim=1) aug conf acc:  0.9126559714795008
RFR (n_estim=1) aug sl acc:  0.8823529411764706
RFR (n_estim=2) base acc:  0.8680926916221033
RFR (n_estim=2) aug acc:  0.8805704099821747
RFR (n_estim=2) aug conf acc:  0.8609625668449198
RFR (n_estim=2) aug sl acc:  0.9055258467023173
RFR (n_estim=3) base acc:  0.8181818181818182
RFR (n_estim=3) aug acc:  0.9019607843137255
RFR (n_estim=3) aug conf acc:  0.8877005347593583
RFR (n_estim=3) aug sl acc:  0.8698752228163993
RFR (n_estim=5) base acc:  0.8912655971479501
RFR (n_estim=5) aug acc:  0.9037433155080213
RFR (n_estim=5) aug conf acc:  0.8912655971479501
RFR (n_estim=5) aug sl acc:  0.8983957219251337
RFR (n_estim=10) base acc:  0.8787878787878788
RFR (n_estim=10) aug acc:  0.8841354723707665
RFR (n_estim=10) aug conf acc:  0.910873440285205
RFR (n_estim=10) aug sl acc:  0.893048128342246
RFR (n_estim=100) base acc:  0.8966131907308378
RFR (

c:\Users\kolan\OneDrive\Pulpit\ml\TabPFN-Distil\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (hidden_l=()) soft-labels aug acc:  0.5490196078431373
MLP (hidden_l=(8,)) base acc:  0.6274509803921569
MLP (hidden_l=(8,)) aug acc:  0.6274509803921569
MLP (hidden_l=(8,)) aug conf acc:  0.7878787878787878
MLP (hidden_l=(8,)) soft-labels aug acc:  0.6666666666666666
MLP (hidden_l=(32,)) base acc:  0.37254901960784315
MLP (hidden_l=(32,)) aug acc:  0.7040998217468806
MLP (hidden_l=(32,)) aug conf acc:  0.6274509803921569
MLP (hidden_l=(32,)) soft-labels aug acc:  0.6274509803921569
MLP (hidden_l=(128,)) base acc:  0.9090909090909091
MLP (hidden_l=(128,)) aug acc:  0.9019607843137255
MLP (hidden_l=(128,)) aug conf acc:  0.37254901960784315
MLP (hidden_l=(128,)) soft-labels aug acc:  0.4563279857397504
MLP (hidden_l=(16, 16)) base acc:  0.37254901960784315
MLP (hidden_l=(16, 16)) aug acc:  0.6987522281639929
MLP (hidden_l=(16, 16)) aug conf acc:  0.38324420677361853
MLP (hidden_l=(16, 16)) soft-labels aug acc:  0.6755793226381461
MLP (hidden_l=(32, 32)) base acc:  0.3725490196078431

,name,base,aug,aug confident,solf-labels aug
0,TabPFN,0.939394,NaN,NaN,NaN
1,RFR n_estim=1,0.811052,0.884135,0.912656,0.882353
2,RFR n_estim=2,0.868093,0.880570,0.860963,0.905526
3,RFR n_estim=3,0.818182,0.901961,0.887701,0.869875
4,RFR n_estim=5,0.891266,0.903743,0.891266,0.898396
5,RFR n_estim=10,0.878788,0.884135,0.910873,0.893048
6,RFR n_estim=100,0.896613,0.903743,0.907308,0.910873
7,MLP hidden_l=(),0.372549,0.627451,0.372549,0.549020
8,"MLP hidden_l=(8,)",0.627451,0.627451,0.787879,0.666667
9,"MLP hidden_l=(32,)",0.372549,0.704100,0.627451,0.627451
